# 02. Coordinates tell a story

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-terrio/blob/18574d4e7a8dabb99d3db65b20c6180185578482/notebooks/02-coordinates.ipynb)



Reading/editing copy only in the shared Atlas kernel. TerrIO 0.9.0 uses a pyemscripten_2025_0 wasm wheel for Pyodide 0.29; the source pins jupyterlite-pyodide-kernel 0.8.0b0 (default Pyodide 0.29.4). This wheel is incompatible with Atlas Pyodide 314; do not install it here or downgrade the shared runtime. [Run this lesson in its matched source Lab](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=02-coordinates.ipynb). The source uses Plotly 6.3.1, xarray 2025.10.1 and optional CDN/WebGL graphics. Natural Earth countries are real generalized geography; all other numerical fixtures are explicitly synthetic. Native HTTP range I/O, specialized drivers and real service/model calls remain optional external continuations.

**Provenance:** `notebooks/02-coordinates.ipynb` at `18574d4e7a8dabb99d3db65b20c6180185578482`; original SHA-256 `c651ef5c580309143c22985d9519872d9bc697fc0625136ac678ac4212e825bb`. Original code MIT; original teaching prose, synthetic data and chapter SVG diagrams CC0. Natural Earth is public domain; TerrIO/dependency/EPSG terms remain separately scoped in retained notices. No TerrIO wheel redistributed.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Chapter 2: Coordinates tell a story; abstract contour illustration](assets/02-coordinates.svg)

# 02. Coordinates tell a story

Units, projections, and honest distances

**Learning goals:** Transform coordinates with TerrIO; check a round trip; quantify Mercator scale distortion.

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays; earlier chapters are helpful but every notebook runs independently.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then **Run → Run All Cells**. First use downloads the runtime and packages. Native Python users install the repository requirements first. Plotly views need network access to its versioned JavaScript CDN and 3D views need WebGL; static figures and numerical summaries remain available.

[Open this chapter in JupyterLite](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=02-coordinates.ipynb) · [Learning resources](https://jltobias.github.io/JupyterLite-terrio/book/resources.html)

**Data:** all generated measurements are synthetic. Natural Earth outlines, where used, are public domain. Generated files go to `outputs/`; download them to retain them outside this browser.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["terrio==0.9.0", "matplotlib", "plotly==6.3.1", "xarray==2025.10.1"])
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import terrio
from terrio_lab import (DATA, OUTPUT, CRS, TRANSFORM, CREDIT, terrain, raster,
                        countries, rings, map_outlines, image_grid, interactive, surface, globe)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready: TerrIO", terrio.__version__, "|", "browser" if sys.platform == "emscripten" else "native Python")

## Longitude first

GeoJSON coordinates use longitude, latitude. EPSG:4326 is angular; UTM is projected. Here we use explicitly invented observation locations near the central meridian of UTM zone 13N. These are teaching positions, not measured sites.

In [ ]:
lon = np.array([-105.02, -105.00, -104.98])
lat = np.array([39.98, 40.00, 40.02])
east, north = terrio.transform_points(lon.tolist(), lat.tolist(), "EPSG:4326", "EPSG:32613")
east, north = np.asarray(east), np.asarray(north)
for row in zip(lon, lat, east, north):
    print("lon %.3f lat %.3f → E %.1f m N %.1f m" % row)

## Check reversibility

A round trip is a useful consistency check. It does not establish survey accuracy or test the suitability of a datum transformation. Real applications also need an area of use, coordinate epoch where relevant, and the required datum grids.

In [ ]:
lon_back, lat_back = terrio.transform_points(east.tolist(), north.tolist(), "EPSG:32613", "EPSG:4326")
np.testing.assert_allclose(lon_back, lon, atol=1e-7, rtol=0)
np.testing.assert_allclose(lat_back, lat, atol=1e-7, rtol=0)
distance = np.hypot(np.diff(east), np.diff(north))
print("Consecutive planar distances (m):", distance.round(1))
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(east, north, "o-", color="#008b8b")
ax.set(xlabel="Easting (m)", ylabel="Northing (m)", title="Invented sites in UTM 13N", aspect="equal")
plt.show()

## Why a web map is not an area calculator

For a spherical Mercator approximation, local scale is sec(latitude), so area scale is sec²(latitude). This teaching formula illustrates the strong latitude dependence. It is not an ellipsoidal distortion analysis for every CRS.

In [ ]:
latitude = np.arange(0, 81)
scale = 1 / np.cos(np.deg2rad(latitude))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(latitude, scale, label="Length scale")
ax.plot(latitude, scale**2, label="Area scale")
ax.set(xlabel="Latitude (degrees)", ylabel="Scale relative to equator", title="Spherical Mercator distortion")
ax.legend(); ax.grid(alpha=.25)
plt.show()
assert np.isclose(scale[60], 2)

## Pixel centres, not corners

A geotransform locates the upper-left corner. Adding half a pixel gives the centre. The row step is negative for this north-up image. These conventions matter when comparing a sampled value to a feature coordinate.

In [ ]:
t = TRANSFORM
col, row = 10, 20
x = t[0] + (col+.5)*t[1] + (row+.5)*t[2]
y = t[3] + (col+.5)*t[4] + (row+.5)*t[5]
print("Pixel centre:", x, y, "in", CRS)
assert x == 500315 and y == 4399385

## Your investigation

At 60° latitude, estimate the Mercator area scale. Explain why a round-trip check cannot certify geodetic accuracy.

### Discussion guide

The spherical area scale is 4. A consistent forward/inverse pair can share the same wrong assumptions or missing datum correction.

## Sources and credit

- [Source 1](https://terrio.dev/crs/)
- [Source 2](https://proj.org/en/stable/operations/projections/merc.html)
- [Source 3](https://epsg.org/terms-of-use.html)

TerrIO: Qiusheng Wu, Mavrick Fitzgerald, Guillermo Morales Sanchez, and contributors (2026). *terrio: Pure-Rust geospatial raster, vector and point cloud data I/O*, v0.9.0. [Zenodo concept DOI](https://doi.org/10.5281/zenodo.23642204). The DOI identifies the project; the execution version is pinned separately.

Natural Earth: Tom Patterson, Nathaniel Vaughn Kelso, and contributors, v5.1.2, [public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/). Original explanations, synthetic data, code, and contour title card: JupyterLite-terrio contributors. Software and underlying third-party notices are documented in the repository's attribution page. Consult that page before redistributing dependency bundles.